# GitOps: how fast is CO2 rising?

You just joined a group that publishes one number: how fast CO2 at Mauna Loa is rising now, in ppm per year. Everything is in Git, so you expect to rerun it and get their number. You can't. The code won't run, and when it does, it doesn't give the number they published.

You'll fix both, and then you'll prove the fix by deleting the environment and the clone and getting the same number back. Keep one idea in mind. The document and the data are the pets, and everything that runs them is cattle.

**The short path takes about 15 minutes:** do step 1, then run `python check.py --short`. Steps 2 and 3 are optional and take about 15 more. Only reading? Each step shows the failure on this page and says how to approach the fix.

Cells that start with `# !` are yours. Remove the `#` when you reach them. Everything else runs as it is.

## Get a copy to work in

This page is the untouched notebook, run once. It shows the failures and stops, because the fixes are yours. To do the work, use your own copy.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-01-gitops?urlpath=lab/tree/gitops.ipynb).** It's free and there's nothing to install. The first start can take a few minutes. A Binder session is temporary and has 2 GB of memory, so download anything you want to keep.

**Or run it on your laptop.** This works on Linux and macOS, on x86_64 or ARM, and on Windows through WSL2. You need [Miniforge](https://github.com/conda-forge/miniforge) or any other conda. If you use mamba, swap it in for `conda`.

```bash
git clone --branch binder-01-gitops --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git gitops
cd gitops
conda env create --file environment.yml
conda activate repro-gitops
jupyter lab gitops.ipynb
```

On Windows, keep the folder in your WSL2 home, not under `/mnt/c`. Git hooks need a Linux file system.

## The repository

This builds the group's repository in `project/`: six commits by three people, and a stand-in for GitHub in `remote/`. Running it again won't touch your work.

In [ ]:
!python scripts/setup_project.py

In [ ]:
!cd project && git log --format='%h  %an: %s'

## Step 1: make it run

The README says to run `python analyze.py`. Try it.

In [ ]:
!cd project && python analyze.py 2>&1 | tail -n 2

Someone installed pandas on their own laptop and started using it. It runs for them and for nobody else. The rule is simple. As you add packages, update the environment file.

There are two environments here. Your session's environment runs Jupyter, and you leave it alone. The project's environment is declared in `project/environment.yml`, and that one is yours to fix and, later, to delete.

Your turn:

1. Open `project/environment.yml` and declare what's missing. Don't pin a version. That's the lock's job.
2. Build the environment from the file. This takes a minute or two.
3. Write the lock. `environment.yml` lists what you asked for, and the lock lists what you got. The lock is named after your platform, because each platform needs its own.
4. Commit both.

In [ ]:
# !conda env create --quiet --prefix envs/co2 --file project/environment.yml
# !conda list --prefix envs/co2 --explicit --md5 > project/explicit-$(conda config --show subdir | cut -d' ' -f2).txt
# !cd project && git add environment.yml explicit-*.txt && git commit -m "Declare pandas and lock the environment"

Now run the analysis in that environment, look at what it did to `metrics.json`, and undo that for now. Don't commit `metrics.json` yet. Step 2 is about why.

In [ ]:
# !cd project && ../envs/co2/bin/python analyze.py
# !cd project && git diff metrics.json
# !cd project && git restore metrics.json

### Check the short path

Run this when step 1 is done. Right now it fails, and each line says what to do next. When it passes, a stranger can build your environment from what's in Git and you've finished the short path. You can stop here.

In [ ]:
!python check.py --short

## Step 2 (optional): find out why the number moved

Here's what the group published, and what a rerun of the committed `params.yaml` and data gives. `check_fresh.py` has been in the repository since the first commit. If the code breaks, the document should break. Here the number stopped matching and nothing told anyone.

In [ ]:
!cd project && cat metrics.json

In [ ]:
!cd project && python check_fresh.py

The tests pass anyway. They test the fit, not the committed number.

In [ ]:
!cd project && python -m pytest -q --color=no 2>&1 | tail -n 1

One commit changed something without a rerun. You could read all six. With 600 you wouldn't. `git bisect run` does a binary search through the history, using any command that exits 0 for good and nonzero for bad. `check_fresh.py` is that kind of command.

Your turn. Note the commit ID it reports, then read that commit with `git show`.

In [ ]:
# !cd project && git bisect start HEAD $(git rev-list --max-parents=0 HEAD)
# !cd project && git bisect run python check_fresh.py
# !cd project && git bisect reset

Now you have to decide, and no tool will do it for you. Should the group publish the old number or the rerun? A higher degree always fits the years it saw a little better, so that comparison can't settle it. `holdout.py` fits up to 2019 and scores each degree on 2020 onward, which the fit never saw. Read the table and pick a degree you can defend.

Your turn:

1. Set `degree` in `project/params.yaml`, rerun, stage, and confirm with `check_fresh.py`. It reads what you staged.
2. Commit. In the message, name the commit `git bisect` found and say why you chose this degree. The next person will read that message instead of asking you.
3. The repository has shipped a hook since its first commit, and nobody installed it. Install it.
4. Try to commit a stale number. Change `degree`, stage it, and commit without a rerun. The hook should refuse. Then put `params.yaml` back.

In [ ]:
# !cd project && python holdout.py
# !cd project && ../envs/co2/bin/python analyze.py && git add params.yaml metrics.json && python check_fresh.py
# !cd project && git commit -m "REPLACE THIS with the commit ID bisect found, the degree you chose, and why"
# !cd project && cp hooks/pre-commit .git/hooks/pre-commit

## Step 3 (optional): delete everything that runs it

Your clone and your environment are cattle. If the project is right, you can throw them away. Push first, because only what's on the remote survives. At the end you tag the release, so people can find exactly this version later.

Your turn. A clone doesn't carry hooks, so you'll need to install yours again. `git status` at the end should show no change to `metrics.json`, which means you got the same number.

In [ ]:
# !cd project && git push origin main
# !rm -rf project envs/co2
# !git clone --config user.name="Workshop Learner" --config user.email="learner@example.org" remote/co2.git project
# !conda create --yes --quiet --prefix envs/co2 --file project/explicit-$(conda config --show subdir | cut -d' ' -f2).txt
# !cd project && ../envs/co2/bin/python analyze.py && git status --short
# !cd project && git tag -a v1.0 -m "Publication version" && git push origin v1.0

## Check everything

This is the full check, for steps 1 to 3. Run it whenever you like. It reads what you committed and tells you what to do next. It knows the right growth rate only as a salted digest, so reading `check.py` won't give you the number. Right now it fails, and that's where you start.

In [ ]:
!python check.py

## Why this CSV isn't in Git LFS

It's 39 KB, and Git handles that fine. Large files are different, because they make every clone heavy. Git LFS keeps them out of the repository, with commands such as `git lfs track "*.pt"` for checkpoints. LFS needs a server. In exercise 2 you'll use DVC, which doesn't.

## On your own project

Look at your last result. Is the `metrics.json` or table you shared the output of what's in Git right now? Add a check like `check_fresh.py`, install it as a hook, and write the lock today.

## About the data

Monthly mean CO2 at Mauna Loa, `data/co2_mm_mlo.csv`, 39 KB, unchanged from [NOAA Global Monitoring Laboratory](https://gml.noaa.gov/ccgg/trends/data.html). NOAA makes the data "freely available to the public and the scientific community" and asks for credit. The terms are in the file's header.

Cite it as: Dr. Xin Lan, NOAA/GML (gml.noaa.gov/ccgg/trends/) and Dr. Ralph Keeling, Scripps Institution of Oceanography (scrippsco2.ucsd.edu/).